# Talks markdown generator for academicpages

Takes a TSV of talks with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `talks.py`. Run either from the `markdown_generator` folder after replacing `talks.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases, rather than Stuart's non-standard TSV format and citation style.

In [38]:
import pandas as pd
import os

## Data format

The TSV needs to have the following columns: title, type, url_slug, venue, date, location, talk_url, description, with a header at the top. Many of these fields can be blank, but the columns must be in the TSV.

- Fields that cannot be blank: `title`, `url_slug`, `date`. All else can be blank. `type` defaults to "Talk" 
- `date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. 
    - The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/talks/YYYY-MM-DD-[url_slug]`
    - The combination of `url_slug` and `date` must be unique, as it will be the basis for your filenames

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [39]:
!cat talks_priya.tsv

title	type	category	collection	url_slug	venue	date	location	talk_url	place	conf
"Concordance structures
of sets of 8 to 10-dimensional manifolds"	Talk	conferences	conferences	Conf_YTM	University of Münster	2024-08-05	Münster, Germany		Germany	Young Topologists Meet-2024
Smooth structures on connected sum of projective spaces	Talk	conferences	conferences	Conf_TA	University of Patras	2023-07-03	Nafpaktos, Greece		Nafpaktos, Greece	International Conference on Topology and its Application-2023
On the stable cohomotopy and KO-groups of connected sum of projective spaces	Talk	conferences	conferences	Conf_RMS	SSN College of Engineering	2022-12-06	Thiruporur, India		Chennai, India	37th Annual Conference of the Ramanujan Mathematical Society
International Conference on Algebra and Analysis	Talk	conferences	conferences	Conf_AA	University of Pune	2017-12-19	Pune, India		Pune, India	In the honor of Prof. S. A. Katre and Dr. Hemant Bhate
Classifying Spaces of Categories	Talk	talks	talks	TRTS	Dept o

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [40]:
talks = pd.read_csv("talks_priya.tsv", sep="\t", header=0)
talks

,title,type,category,collection,url_slug,venue,date,location,talk_url,place,conf
0,Concordance structures\nof sets of 8 to 10-dim...,Talk,conferences,conferences,Conf_YTM,University of Münster,2024-08-05,"Münster, Germany",NaN,Germany,Young Topologists Meet-2024
1,Smooth structures on connected sum of projecti...,Talk,conferences,conferences,Conf_TA,University of Patras,2023-07-03,"Nafpaktos, Greece",NaN,"Nafpaktos, Greece",International Conference on Topology and its A...
2,On the stable cohomotopy and KO-groups of conn...,Talk,conferences,conferences,Conf_RMS,SSN College of Engineering,2022-12-06,"Thiruporur, India",NaN,"Chennai, India",37th Annual Conference of the Ramanujan Mathem...
3,International Conference on Algebra and Analysis,Talk,conferences,conferences,Conf_AA,University of Pune,2017-12-19,"Pune, India",NaN,"Pune, India",In the honor of Prof. S. A. Katre and Dr. Hema...
4,Classifying Spaces of Categories,Talk,talks,talks,TRTS,"Dept of Mathematics, IIT Bombay",2024-02-07,"Powai, India",NaN,"Mumbai, India",Topology and Related Topics Seminar
5,Quillen equivalent stable homotopy categories,Talk,talks,talks,TRTS,"Dept of Mathematics, IIT Bombay",2023-08-20,"Powai, India",NaN,"Mumbai, India",Topology and Related Topics Seminar
6,A brief introduction to the stable homotopy ca...,Talk,talks,talks,TRTS,"Dept of Mathematics, IIT Bombay",2023-08-15,"Powai, India",NaN,"Mumbai, India",Topology and Related Topics Seminar
7,Essential Categorical Terms for Stable Homotop...,Talk,talks,talks,TRTS,"Dept of Mathematics, IIT Bombay",2023-08-01,"Powai, India",NaN,"Mumbai, India",Topology and Related Topics Seminar
8,Introduction to Spectra and Stable Homotopy Ca...,Talk,talks,talks,DuTA,"ICTS, TIFR Bengaluru",2023-05-12,"Bangalore, India",NaN,"Bangalore, India",Dualities in Topology and Algebra 2023
9,Spectra and stable homotopy category,Talk,talks,talks,Tt-cat,Online seminar series,2023-03-12,"Powai, India",NaN,Online,tt-category and Chromatic Homotopy Theory


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [41]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    if type(text) is str:
        return "".join(html_escape_table.get(c,c) for c in text)
    else:
        return "False"

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [42]:
loc_dict = {}

for row, item in talks.iterrows():
    
    md_filename = str(item.date) + "-" + item.url_slug + ".md"
    html_filename = str(item.date) + "-" + item.url_slug 
    year = item.date[:4]
    
    md = "---\ntitle: \""   + item.title + '"\n'
    md += "collection: talks" + "\n"
    
    if len(str(item.type)) > 3:
        md += 'type: "' + item.type + '"\n'
    else:
        md += 'type: "Talk"\n'
    
    md += "permalink: /talks/" + html_filename + "\n"
    
    if len(str(item.venue)) > 3:
        md += 'venue: "' + item.venue + '"\n'
        
    if len(str(item.date)) > 3:
        md += "date: " + str(item.date) + "\n"
    
    if len(str(item.location)) > 3:
        md += 'location: "' + str(item.location) + '"\n'
    
    if len(str(item.category)) > 3:
        md += 'category: "' + str(item.category) + '"\n'
    
    if len(str(item.place)) > 3:
        md += 'place: "' + str(item.place) + '"\n'
    
    if len(str(item.conf)) > 3:
        md += 'name: "' + str(item.conf) + '"\n'

    
    md += "---\n"
    
    
    if len(str(item.talk_url)) > 3:
        md += "\n[More information here](" + item.talk_url + ")\n" 
        
        
    md_filename = os.path.basename(md_filename)
    #print(md)
    
    with open("../_talks/" + md_filename, 'w') as f:
        f.write(md)

These files are in the talks directory, one directory below where we're working from.

In [43]:
!ls ../_talks

2017-06-14-AFS-I.md	2023-05-12-DuTA.md     2024-02-07-TRTS.md
2017-12-19-Conf_AA.md	2023-05-15-ICTS-DA.md  2024-08-05-Conf_YTM.md
2018-11-12-AFS-II.md	2023-07-03-Conf_TA.md  2024-11-28-NCM.md
2019-01-01-DJS.md	2023-07-13-NCM.md      talkmap.ipynb
2020-02-08-GHT.md	2023-08-01-TRTS.md     talkmap.py
2021-02-11-ICTS-DA2.md	2023-08-15-TRTS.md     talks.ipynb
2022-12-06-Conf_RMS.md	2023-08-20-TRTS.md     Untitled.ipynb
2023-03-12-Tt-cat.md	2023-09-01-SS.md
2023-04-12-SS.md	2023-12-04-NCM.md


In [44]:
!cat ../_talks/2023-04-12-SS.md

---
title: "A brief introduction to the fiber bundle"
collection: talks
type: "Talk"
permalink: /talks/2023-04-12-SS
venue: "Dept of Mathematics, IIT Bombay"
date: 2023-04-12
location: "Powai, India"
category: "expository"
place: "Mumbai, India"
name: "Student Seminar 19"
---
